In [1]:
##Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [2]:
#Loading the Dataset
DATA_PATH = "../data/raw/netflix_titles.csv"   # because notebook is inside notebooks folder

df = pd.read_csv(DATA_PATH)
df.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
print("Shape:", df.shape)
df.info()


Shape: (8807, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [4]:
#Missing Values Check
df.isnull().sum()


show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [5]:
#Duplicate Check
print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [6]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.columns


Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

In [7]:
text_cols = ["title", "director", "cast", "country", "rating", "duration", "listed_in"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()


In [8]:
df.replace("nan", np.nan, inplace=True)
df.isnull().sum()


show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [9]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["date_added"].head()


0   2021-09-25
1   2021-09-24
2   2021-09-24
3   2021-09-24
4   2021-09-24
Name: date_added, dtype: datetime64[ns]

In [10]:
#Feature Engineering

df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["genre_count"] = df["listed_in"].apply(lambda x: len(str(x).split(",")) if pd.notnull(x) else 0)
df["country_count"] = df["country"].apply(lambda x: len(str(x).split(",")) if pd.notnull(x) else 0)


In [11]:
#Duration Split (Movie vs TV Show)
df["duration_value"] = df["duration"].str.extract("(\d+)").astype(float)

df["duration_type"] = df["duration"].apply(
    lambda x: "minutes" if "min" in str(x).lower() else ("seasons" if "season" in str(x).lower() else np.nan)
)

df[["type", "duration", "duration_value", "duration_type"]].head(10)


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Varshitha Samiappan\AppData\Local\Temp\ipykernel_24192\3306485933.py:2: SyntaxWarning: invalid escape sequence '\d'
  df["duration_value"] = df["duration"].str.extract("(\d+)").astype(float)


,type,duration,duration_value,duration_type
0,Movie,90 min,90.0,minutes
1,TV Show,2 Seasons,2.0,seasons
2,TV Show,1 Season,1.0,seasons
3,TV Show,1 Season,1.0,seasons
4,TV Show,2 Seasons,2.0,seasons
5,TV Show,1 Season,1.0,seasons
6,Movie,91 min,91.0,minutes
7,Movie,125 min,125.0,minutes
8,TV Show,9 Seasons,9.0,seasons
9,Movie,104 min,104.0,minutes


In [12]:
#Handle Missing Values

df["director"].fillna("Unknown", inplace=True)
df["cast"].fillna("Unknown", inplace=True)
df["country"].fillna("Unknown", inplace=True)
df["rating"].fillna("Unknown", inplace=True)
df["duration"].fillna("Unknown", inplace=True)
df["year_added"].fillna(df["release_year"], inplace=True)


C:\Users\Varshitha Samiappan\AppData\Local\Temp\ipykernel_24192\1968199637.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["director"].fillna("Unknown", inplace=True)
C:\Users\Varshitha Samiappan\AppData\Local\Temp\ipykernel_24192\1968199637.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always b

In [13]:
df.isnull().sum()


show_id            0
type               0
title              0
director           0
cast               0
country            0
date_added        98
release_year       0
rating             0
duration           0
listed_in          0
description        0
year_added         0
month_added       98
genre_count        0
country_count      0
duration_value     3
duration_type      3
dtype: int64

In [14]:
# Fix rating column (remove duration values accidentally inside rating)
df["rating"] = df["rating"].astype(str).str.strip()

# Remove rows where rating contains "min" or "Season"
df.loc[df["rating"].str.contains("min", case=False, na=False), "rating"] = None
df.loc[df["rating"].str.contains("Season", case=False, na=False), "rating"] = None


In [15]:
df["rating"].value_counts().head(20)


rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
Unknown        4
NC-17          3
UR             3
Name: count, dtype: int64

In [16]:
df[df["rating"].astype(str).str.contains("min", na=False)]


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,genre_count,country_count,duration_value,duration_type


In [21]:
df.to_csv("data/processed/netflix_cleaned.csv", index=False)


In [18]:
# Fix wrong rating values (like '66 min') moved into rating column
df["rating"] = df["rating"].astype(str).str.strip()

# If rating contains "min" or "Season", replace it with "NR"
df.loc[df["rating"].str.contains("min|Season", case=False, na=False), "rating"] = "NR"


In [19]:
OUTPUT_PATH = "../data/processed/netflix_processed.csv"

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)

print("✅ Saved processed dataset at:", OUTPUT_PATH)
print("Final Shape:", df.shape)


✅ Saved processed dataset at: ../data/processed/netflix_processed.csv
Final Shape: (8807, 18)


In [20]:
df.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,genre_count,country_count,duration_value,duration_type
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0,1,1,90.0,minutes
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0,3,1,2.0,seasons
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0,3,0,1.0,seasons
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0,2,0,1.0,seasons
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0,3,1,2.0,seasons
